# Creation of Crop Yield Estimation Model Using Geo-Spatial Technology
**B.Tech Thesis Project** | Department of Geo-Engineering & RDT, Andhra University College of Engineering  

**Author:** Baipilli Naveen & Team (Avupati Ajay, Marrapu Sai Meghana, Pothuraju Pooja, Devu Phanindra Sai Kumar)  
**Project Guide:** Prof. Murali Krishna Gurram  

---
### Project Overview
This notebook demonstrates the end-to-end geoprocessing pipeline for estimating regional rice crop yield (kg/ha) using single-date Landsat 8 satellite data (NDVI) and empirical exponential modeling ($y = 1841.6 \cdot e^{2.5262x}$, $R^2 = 0.9922$).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.indices.ndvi_calculator import calculate_ndvi, extract_crop_ndvi
from src.modeling.yield_model import (
    estimate_yield_from_ndvi,
    calculate_yield_statistics,
    compute_percentage_accuracy
)
from src.config import GROUND_TRUTH, YIELD_MODEL_COEFFICIENT_A, YIELD_MODEL_COEFFICIENT_B

print("Project modules loaded successfully!")

## 1. Ground Truth Agricultural Statistics
Data collected from Ministry of Agriculture & Farmers Welfare (DAC&FW) and Directorate of Economics and Statistics (DES), AP for Rabi season 2019–2020.

In [ ]:
stats_path = project_root / 'data' / 'raw' / 'statistics' / 'crop_statistics_ap_rabi_2019_2020.csv'
df_stats = pd.read_csv(stats_path)
df_stats

## 2. Exponential Yield Regression Model
During heading and peak flowering stages (February 23, 2020), rice canopy NDVI exhibits strong correlation with final grain yield:

$$\text{Yield} = 1841.6 \times e^{2.5262 \times \text{NDVI}} \quad (R^2 = 0.9922)$$

In [ ]:
ndvi_range = np.linspace(0.15, 0.65, 200)
yield_curve = estimate_yield_from_ndvi(ndvi_range)

plt.figure(figsize=(9, 5))
plt.plot(ndvi_range, yield_curve, label='Fitted Yield Model (R² = 0.9922)', color='green', linewidth=2.5)
plt.scatter([0.285720, 0.408456, 0.531192], [3790.21, 5333.55, 7046.50], color='red', s=60, zorder=5, label='Thesis Observation Points (Min, Mean, Max)')
plt.title('Relationship Between Mature Rice NDVI and Crop Yield', fontsize=12, fontweight='bold')
plt.xlabel('Rice Crop NDVI', fontsize=11)
plt.ylabel('Estimated Crop Yield (kg/ha)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3. Spatial Simulation & Paddy Yield Map Generation
Simulating a 2D spatial raster grid of Godavari delta paddy fields and extracting minimum, maximum, and mean yield statistics.

In [ ]:
np.random.seed(42)
rows, cols = 120, 120
sim_nir = np.random.normal(0.52, 0.08, (rows, cols))
sim_red = np.random.normal(0.14, 0.03, (rows, cols))

sim_ndvi = calculate_ndvi(sim_nir, sim_red)
sim_yield = estimate_yield_from_ndvi(sim_ndvi)
stats = calculate_yield_statistics(sim_yield)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im1 = axes[0].imshow(sim_ndvi, cmap='YlGn', vmin=0.2, vmax=0.6)
axes[0].set_title('Simulated Rice Canopy NDVI', fontsize=12)
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(sim_yield, cmap='RdYlGn', vmin=3500, vmax=7500)
axes[1].set_title('Predicted Crop Yield (kg/ha)', fontsize=12)
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"Minimum Estimated Yield: {stats['min_yield']:.2f} kg/ha")
print(f"Maximum Estimated Yield: {stats['max_yield']:.2f} kg/ha")
print(f"Mean Estimated Yield:    {stats['mean_yield']:.2f} kg/ha")

## 4. Accuracy Verification
Evaluation against official district census data:

In [ ]:
eg_area_acc = compute_percentage_accuracy(GROUND_TRUTH['East Godavari']['observed_area_ha'], GROUND_TRUTH['East Godavari']['actual_area_ha'])
eg_yield_acc = compute_percentage_accuracy(GROUND_TRUTH['East Godavari']['observed_yield_kg_per_ha'], GROUND_TRUTH['East Godavari']['actual_yield_kg_per_ha'])
wg_yield_acc = compute_percentage_accuracy(GROUND_TRUTH['West Godavari']['observed_yield_kg_per_ha'], GROUND_TRUTH['West Godavari']['actual_yield_kg_per_ha'])

results_table = pd.DataFrame([
    {'Study Area': 'East Godavari (Model Creation)', 'Target': 'Paddy Area (Ha)', 'Actual': 166544.0, 'Observed': 150485.8, 'Accuracy': f'{eg_area_acc:.2f}%'},
    {'Study Area': 'East Godavari (Model Creation)', 'Target': 'Paddy Yield (kg/ha)', 'Actual': 5333.55, 'Observed': 5418.36, 'Accuracy': f'{eg_yield_acc:.2f}%'},
    {'Study Area': 'West Godavari (Model Validation)', 'Target': 'Paddy Yield (kg/ha)', 'Actual': 5394.66, 'Observed': 5379.22, 'Accuracy': f'{wg_yield_acc:.2f}%'}
])
results_table